# Classification de Races de Chiens
## Stanford Dogs Dataset — 20 580 images, 120 races

Dans ce notebook nous verrons les différentes étapes qui menerons à la classification des races de chiens:
1. Téléchargement et organisation du dataset
2. Exploration et visualisation des données -> à des fins statistiques pour évaluer les données présentes dans le DATA SET  (redimensionnement ?, nombre d'images par sous dossier, etc.)
3. Préparation des images
4. Entraînement d'un modèle de classification
5. Évaluation et résultats

Notes
- L'entraînement complet peut prendre plusieurs heures.

## 0. Installation des bibliothèques

In [ ]:
# Commande pour installer les bibliothèques nécessaires à l'entrainement du modèle
pip install numpy pandas matplotlib seaborn scikit-learn opencv-python Pillow tqdm

## 1. Téléchargement du Dataset Stanford Dogs

In [ ]:
import os
import urllib.request
import tarfile
import subprocess
import sys

# Author CHAKER Zakaria & NSANGUE Nathan
# Dans cette section nous récupérons le dataset présent en ligne...
# Ce script permet de télécharger et d'extraire le dataset en utilisant des bibliothèques adaptées
# comme tarfile pour l'extraction.
# Une vérification de la connexion Internet est effectuée avant le téléchargement.
# Gestion des erreurs ajoutée pour améliorer la robustesse.
# Version 1.1

BASE_DIR = "./stanford_dogs"
os.makedirs(BASE_DIR, exist_ok=True)

# Chargement des URL dans des variables 
IMAGES_URL = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
ANNOTS_URL = "http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar"
LISTS_URL  = "http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar"

def check_internet():
    """
    Vérifie la connexion Internet en envoyant un ping vers 8.8.8.8.
    Retourne True si Internet est accessible, sinon False.
    """
    try:
        param = "-n" if sys.platform.lower() == "win32" else "-c"
        result = subprocess.run(
            ["ping", param, "1", "8.8.8.8"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        return result.returncode == 0
    except Exception as e:
        print("Erreur lors de la vérification de la connexion Internet :", e)
        return False

def download_and_extract(url, dest_dir):
    """
    Télécharge et décompresse un fichier .tar
    """
    filename = os.path.join(dest_dir, url.split("/")[-1])

    # Téléchargement
    if not os.path.exists(filename):
        print(f"Téléchargement de {url.split('/')[-1]}...")
        try:
            urllib.request.urlretrieve(url, filename)
            print("Téléchargement terminé.")
        except Exception as e:
            print("Erreur lors du téléchargement :", e)
            return
    else:
        print(f"{url.split('/')[-1]} déjà présent, on passe.")

    # Extraction des archives
    print("Extraction en cours...")
    try:
        with tarfile.open(filename) as tar:
            tar.extractall(dest_dir)
        print("Extraction terminée.")
    except Exception as e:
        print("Erreur lors de l'extraction :", e)

# Vérification de la connexion Internet...
if not check_internet():
    print("Aucune connexion Internet détectée. Veuillez vérifier votre réseau.")
    sys.exit(1)

download_and_extract(IMAGES_URL, BASE_DIR)
download_and_extract(LISTS_URL, BASE_DIR)

print("\nDataset prêt.")

## 2. Exploration du Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.io import loadmat
import glob

# Author CHAKER Zakaria & NSANGUE Nathan
# TODO
# Version 1.1

# Charger les fichiers de listes (train/test splits officiels)
IMAGES_DIR = os.path.join(BASE_DIR, "Images")

# Lister toutes les races (= sous-dossiers)
breed_folders = sorted([
    d for d in os.listdir(IMAGES_DIR)
    if os.path.isdir(os.path.join(IMAGES_DIR, d))
])

print(f"Nombre de races : {len(breed_folders)}")

In [ ]:
import glob
import pandas as pd
# Metre à jour les import et commentaires et corriger 
#TODO
# Compter le nombre d'images par race
breed_counts = {}
for folder in breed_folders:
    path = os.path.join(IMAGES_DIR, folder)
    images = glob.glob(os.path.join(path, "*.jpg"))
    name = folder.split("-", 1)[-1].replace("_", " ").title()
    breed_counts[name] = len(images)

df_counts = pd.DataFrame.from_dict(breed_counts, orient="index", columns=["nb_images"])
df_counts = df_counts.sort_values("nb_images", ascending=False)

print("Statistiques globales :")
print(f"Total images     : {df_counts['nb_images'].sum()}")
print(f"  Races            : {len(df_counts)}")
print(f"  Moy. par race    : {df_counts['nb_images'].mean():.1f}")
print(f"  Min. par race    : {df_counts['nb_images'].min()}")
print(f"  Max. par race    : {df_counts['nb_images'].max()}")

In [ ]:
# Metre à jour les import et commentaires et corriger 
#TODO

# Visualisation : distribution du nombre d'images par race
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Top 20 races les plus représentées
top20 = df_counts.head(20)
axes[0].barh(top20.index[::-1], top20["nb_images"][::-1], color="steelblue")
axes[0].set_title("Top 20 races les plus représentées", fontsize=13)
axes[0].set_xlabel("Nombre d'images")
axes[0].axvline(df_counts["nb_images"].mean(), color="red", linestyle="--", label="Moyenne")
axes[0].legend()

# Distribution globale (histogramme)
axes[1].hist(df_counts["nb_images"], bins=20, color="coral", edgecolor="white")
axes[1].set_title("Distribution du nb d'images par race", fontsize=13)
axes[1].set_xlabel("Nombre d'images")
axes[1].set_ylabel("Nombre de races")
axes[1].axvline(df_counts["nb_images"].mean(), color="red", linestyle="--", label="Moyenne")
axes[1].legend()

plt.tight_layout()
plt.savefig("distribution_races.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graphique sauvegardé : distribution_races.png")

In [ ]:
# Import à MAJ et commentaires et les variables 
# Analyse des dimensions d'images (échantillon de 500 images)
print("Analyse des dimensions (échantillon 500 images)...")
widths, heights = [], []

all_images = glob.glob(os.path.join(IMAGES_DIR, "**", "*.jpg"), recursive=True)
sample_imgs = np.random.choice(all_images, min(500, len(all_images)), replace=False)

for img_path in sample_imgs:
    try:
        with Image.open(img_path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except:
        pass

print(f"\nDimensions des images :")
print(f"  Largeur  — min: {min(widths)}, max: {max(widths)}, moy: {np.mean(widths):.0f}")
print(f"  Hauteur  — min: {min(heights)}, max: {max(heights)}, moy: {np.mean(heights):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color="teal", edgecolor="white")
axes[0].set_title("Distribution des largeurs")
axes[0].set_xlabel("Pixels")
axes[1].hist(heights, bins=30, color="orange", edgecolor="white")
axes[1].set_title("Distribution des hauteurs")
axes[1].set_xlabel("Pixels")
plt.tight_layout()
plt.savefig("dimensions_images.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 🔧 3. Préparation des Données

On va :
- Redimensionner toutes les images à **128×128** (compromis qualité/vitesse pour CPU)
- Normaliser les pixels entre 0 et 1
- Séparer en train / validation / test
- Créer les labels (encodage numérique des races)

> 💡 **Astuce CPU** : On utilise 128×128 au lieu de 224×224. Pour un meilleur modèle, augmenter à 224×224 si tu as plus de temps.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

IMG_SIZE = 128   # Taille cible (128x128 pixels)
MAX_PER_CLASS = 80  # Limite par race pour rester rapide sur CPU
                    # Mettre None pour utiliser toutes les images

print(f"Paramètres :")
print(f"  Taille image : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Max par race : {MAX_PER_CLASS}")
print(f"  Races        : {len(breed_folders)}")
print(f"\nChargement des images (patience...)")

X_data = []  # Images
y_data = []  # Labels (noms de races)

for folder in tqdm(breed_folders, desc="Chargement"):
    path = os.path.join(IMAGES_DIR, folder)
    imgs = glob.glob(os.path.join(path, "*.jpg"))
    
    if MAX_PER_CLASS:
        imgs = imgs[:MAX_PER_CLASS]
    
    name = folder.split("-", 1)[-1]  # Nom de la race
    
    for img_path in imgs:
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize((IMG_SIZE, IMG_SIZE))
            X_data.append(np.array(img))
            y_data.append(name)
        except Exception as e:
            pass  # Ignore les images corrompues

X_data = np.array(X_data, dtype=np.float32) / 255.0  # Normalisation 0-1
y_data = np.array(y_data)

print(f"\n✅ Données chargées !")
print(f"  X shape : {X_data.shape}  ({X_data.nbytes / 1e6:.0f} MB)")
print(f"  y shape : {y_data.shape}")

In [ ]:
# Encoder les labels (texte → entiers)
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)

print(f"Classes encodées : {len(le.classes_)} races")
print(f"Exemples : {list(zip(le.classes_[:5], range(5)))}")

# Split 70% train / 15% validation / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\n📂 Répartition des données :")
print(f"  Train      : {len(X_train)} images ({len(X_train)/len(X_data)*100:.0f}%)")
print(f"  Validation : {len(X_val)} images ({len(X_val)/len(X_data)*100:.0f}%)")
print(f"  Test       : {len(X_test)} images ({len(X_test)/len(X_data)*100:.0f}%)")

---
## 🧠 4. Modèle de Classification

On utilise deux approches :

**A) Baseline rapide** — Random Forest sur HOG features (adapté CPU)

**B) SVM avec features** — SVM + PCA + HOG (meilleure précision)

> Pour un vrai réseau de neurones (bien meilleur), il faudrait PyTorch ou TensorFlow avec GPU.

In [ ]:
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time

def extract_hog_features(images, desc="HOG"):
    """Extrait les features HOG de chaque image."""
    features = []
    for img in tqdm(images, desc=desc):
        img_uint8 = (img * 255).astype(np.uint8)
        feat = hog(
            img_uint8,
            orientations=8,
            pixels_per_cell=(16, 16),
            cells_per_block=(2, 2),
            channel_axis=-1
        )
        features.append(feat)
    return np.array(features)

print("Extraction des features HOG (train)...")
X_train_hog = extract_hog_features(X_train, "Train HOG")
print("Extraction des features HOG (val)...")
X_val_hog   = extract_hog_features(X_val,   "Val HOG")
print("Extraction des features HOG (test)...")
X_test_hog  = extract_hog_features(X_test,  "Test HOG")

print(f"\nShape features HOG : {X_train_hog.shape}")

In [ ]:
# ---- Modèle A : Random Forest (Baseline) ----
print("=" * 50)
print("Modèle A — Random Forest (Baseline)")
print("=" * 50)

t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    n_jobs=-1,        # Utilise tous les cœurs CPU
    random_state=42
)
rf.fit(X_train_hog, y_train)
t1 = time.time()

y_pred_rf = rf.predict(X_val_hog)
acc_rf = accuracy_score(y_val, y_pred_rf)

print(f"  Temps d'entraînement : {t1-t0:.1f}s")
print(f"  Accuracy (validation) : {acc_rf*100:.1f}%")

In [ ]:
# ---- Modèle B : SVM + PCA (meilleure précision) ----
print("=" * 50)
print("Modèle B — LinearSVC + PCA")
print("=" * 50)

pipeline_svm = Pipeline([
    ("pca",  PCA(n_components=200, random_state=42)),  # Réduction de dimension
    ("svm",  LinearSVC(C=1.0, max_iter=2000, random_state=42))
])

t0 = time.time()
pipeline_svm.fit(X_train_hog, y_train)
t1 = time.time()

y_pred_svm = pipeline_svm.predict(X_val_hog)
acc_svm = accuracy_score(y_val, y_pred_svm)

print(f"  Temps d'entraînement : {t1-t0:.1f}s")
print(f"  Accuracy (validation) : {acc_svm*100:.1f}%")

---
## 📊 5. Évaluation et Résultats

In [ ]:
# Évaluation finale sur le jeu de TEST avec le meilleur modèle
best_model = pipeline_svm if acc_svm >= acc_rf else rf
best_name  = "LinearSVC+PCA" if acc_svm >= acc_rf else "RandomForest"

y_pred_test = best_model.predict(X_test_hog)
acc_test = accuracy_score(y_test, y_pred_test)

print(f"🏆 Meilleur modèle : {best_name}")
print(f"   Accuracy sur TEST : {acc_test*100:.1f}%")
print()

# Rapport détaillé (Top 10 races)
print("Rapport de classification (extrait — 10 premières races) :")
report = classification_report(
    y_test, y_pred_test,
    target_names=le.classes_,
    output_dict=True
)
df_report = pd.DataFrame(report).T
df_report = df_report.iloc[:-3]  # Enlever les lignes de summary
df_report = df_report.sort_values("f1-score", ascending=False)
print(df_report[["precision", "recall", "f1-score", "support"]].head(10).to_string())

In [ ]:
# Matrice de confusion (top 20 races)
top20_breeds = df_report.head(20).index.tolist()
top20_idx    = [list(le.classes_).index(b) for b in top20_breeds if b in list(le.classes_)]

mask = np.isin(y_test, top20_idx)
cm   = confusion_matrix(y_test[mask], y_pred_test[mask], labels=top20_idx)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=False, fmt="d", cmap="Blues",
    xticklabels=[b[:15] for b in top20_breeds],
    yticklabels=[b[:15] for b in top20_breeds],
    ax=ax
)
ax.set_title("Matrice de Confusion — Top 20 races", fontsize=14)
ax.set_xlabel("Prédit")
ax.set_ylabel("Réel")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("📷 Matrice sauvegardée : confusion_matrix.png")

In [ ]:
# Comparaison des modèles
fig, ax = plt.subplots(figsize=(7, 4))
models  = ["Random Forest", "LinearSVC + PCA"]
scores  = [acc_rf * 100, acc_svm * 100]
colors  = ["steelblue", "coral"]

bars = ax.bar(models, scores, color=colors, edgecolor="white", width=0.4)
ax.set_ylim(0, 100)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Comparaison des modèles — Validation")
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{score:.1f}%", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.savefig("comparaison_modeles.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Prédictions sur quelques images test
n_samples = 10
indices   = np.random.choice(len(X_test), n_samples, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, idx in enumerate(indices):
    axes[i].imshow(X_test[idx])
    true_label = le.classes_[y_test[idx]].replace("_", " ").title()
    pred_label = le.classes_[y_pred_test[idx]].replace("_", " ").title()
    color = "green" if y_test[idx] == y_pred_test[idx] else "red"
    axes[i].set_title(f"Réel : {true_label[:20]}\nPrédit : {pred_label[:20]}",
                      color=color, fontsize=8)
    axes[i].axis("off")

plt.suptitle("Exemples de prédictions (vert=correct, rouge=erreur)", fontsize=13)
plt.tight_layout()
plt.savefig("predictions_exemples.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 💾 6. Sauvegarde du Modèle

In [ ]:
import pickle

# Sauvegarder le meilleur modèle + le label encoder
with open("best_model.pkl", "wb") as f:
    pickle.dump({"model": best_model, "label_encoder": le}, f)

print(f"✅ Modèle sauvegardé : best_model.pkl")
print(f"   Pour recharger : pickle.load(open('best_model.pkl','rb'))")

---
## 🔮 7. Utiliser le Modèle sur une Nouvelle Image

In [ ]:
def predict_breed(image_path, model, label_encoder, img_size=128):
    """
    Prédit la race d'un chien à partir du chemin d'une image.
    
    Args:
        image_path : chemin vers l'image (ex: 'mon_chien.jpg')
        model      : modèle entraîné
        label_encoder : LabelEncoder utilisé pendant l'entraînement
    
    Returns:
        Nom de la race prédite
    """
    # Charger et préparer l'image
    img = Image.open(image_path).convert("RGB")
    img = img.resize((img_size, img_size))
    img_array = np.array(img, dtype=np.float32) / 255.0
    
    # Extraire les features HOG
    img_uint8 = (img_array * 255).astype(np.uint8)
    features = hog(img_uint8, orientations=8,
                   pixels_per_cell=(16, 16), cells_per_block=(2, 2),
                   channel_axis=-1)
    
    # Prédire
    pred_idx  = model.predict([features])[0]
    pred_race = label_encoder.classes_[pred_idx].replace("_", " ").title()
    
    return pred_race

# Exemple d'utilisation :
# race = predict_breed("mon_chien.jpg", best_model, le)
# print(f"Race prédite : {race}")

print("✅ Fonction predict_breed() prête à l'emploi !")
print("   Utilise : predict_breed('chemin/vers/image.jpg', best_model, le)")

---
## 📌 Résumé et Pistes d'Amélioration

### Ce qu'on a fait :
- ✅ Téléchargement du Stanford Dogs Dataset (120 races)
- ✅ Exploration : distribution, exemples visuels, dimensions
- ✅ Préparation : redimensionnement, normalisation, split train/val/test
- ✅ Features HOG (Histogram of Oriented Gradients)
- ✅ Deux modèles : Random Forest et LinearSVM+PCA
- ✅ Évaluation : accuracy, rapport, matrice de confusion

### Pour aller plus loin :

| Amélioration | Gain estimé |
|---|---|
| Utiliser un réseau de neurones (CNN) | +30 à +50% accuracy |
| Transfer Learning (ResNet, MobileNet) | Jusqu'à 80-90% accuracy |
| Augmentation de données | +5 à +10% |
| Taille image 224×224 | +5% (besoin GPU) |
| Toutes les images sans limite MAX_PER_CLASS | +5 à +15% |